# 42장 · 실행 근거를 모아 프로젝트에 들어가기

동료가 ‘답은 14일이라고 나오는데 화면 근거는 휴관 안내예요’라고 보고했습니다. 프로젝트에 들어가기 전, 질문 한 건을 따라가 어디부터 다시 확인할지 정합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 도구 요청과 화면 출력 사이 좁혀 가기

정상 응답을 한꺼번에 다시 만들면 증상을 찾기 어렵습니다. 질문을 고정한 뒤 검색기가 돌려준 ID, 답이 인용한 ID, 화면이 실제로 띄운 출처를 순서대로 읽습니다. 아래는 불일치를 재현한 수업용 화면 기록입니다.


### 셀 01 · 검색 직후 근거 확인하기

자료 세 쪽에서 질문 한 건을 다시 실행해 처음 찾은 문서를 읽습니다.

화면에 보여 줄 출처는 이번 질문에서 받은 문서만 사용하세요. 검색 결과가 비었거나 답에 붙은 인용 ID가 검색 문서와 다르면 현재 출처를 비우고 확인을 요청합니다. 지난 질문의 출처를 대신 붙이지 않습니다.

In [1]:
records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
terms = ("대출", "휴관", "반납")

def retrieve(question, k=1):
    query = [int(term in question) for term in terms]
    if not any(query):
        return []
    def distance(row):
        vector = [int(term in row["text"]) for term in terms]
        return sum((a-b)**2 for a, b in zip(query, vector)), row["id"]
    return sorted(records, key=distance)[:k]

question = '대출 기간은 며칠인가요?'
found = retrieve(question, 1)
if found:
    print(found[0]['id'], found[0]['text'])
else:
    print('이번 질문의 검색 결과가 없습니다.')

library-1 도서 대출 기간은 14일입니다.


검색 직후에는 library-1과 14일이 맞아떨어집니다. 검색 입력을 바꾸기 전에 답과 화면 사이를 살펴볼 차례입니다.

### 셀 02 · 화면에 남은 지난번 출처 확인하기

같은 질문의 검색 ID와 화면의 이전 출처를 대조하세요.

In [2]:
screen = {'answer': '도서 대출 기간은 14일입니다. [library-1]', 'source': records[1]}
print(screen['source']['id'], screen['source']['page'])

library-2 2


화면은 library-2의 2쪽을 가리킵니다. 답의 인용과 검색은 대출인데 화면에만 지난 휴관 문서가 남았습니다.

## 같은 질문으로 수정 결과 확인하기

검색 방식을 바꾸거나 모델에 다시 요청할 필요가 있는지 고르세요. 현재 실행 결과의 출처를 표시하는 흐름만 고치면 같은 입력으로 증상을 확인할 수 있습니다.


### 과제 03 · 검색 결과의 출처 다시 연결하기

답에서 인용을 읽고 모두 이번 검색 결과에 있는지 확인하세요. 맞으면 이 예제의 첫 인용을 화면에 연결하세요. 문서가 없거나 인용이 빠지거나 틀리면 출처를 비우고 답 자리에는 안내를 표시하세요.

In [3]:
# 답변에 현재 문서의 ID가 실제로 인용됐는지 확인한 뒤 교체하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>import re

def match_source(answer, documents):
    cited = re.findall(r&quot;\[([^\[\]]+)\]&quot;, answer)
    by_id = {doc[&quot;id&quot;]: doc for doc in documents}
    if cited and all(key in by_id for key in cited):
        return by_id[cited[0]]
    return None

source = match_source(screen[&quot;answer&quot;], found)
if source is None:
    screen = {&quot;answer&quot;: &quot;이번 답의 근거를 확인할 수 없습니다.&quot;, &quot;source&quot;: None}
    print(screen[&quot;answer&quot;])
else:
    screen = {**screen, &quot;source&quot;: source}
    print(source[&quot;id&quot;], source[&quot;page&quot;])</code></pre>
<p>실행 결과</p><pre><code>library-1 1</code></pre>
<p>기본 질문에서는 library-1 1이 보입니다. 셀 01 질문을 ‘주차 요금은?’으로 바꾸고 01~03을 다시 실행하세요. 이번 답의 근거를 확인할 수 없다는 문구가 나오고 screen[&quot;source&quot;]는 None이 됩니다.</p>
</details>

## 돌아보기

동료에게 재현 질문, 처음 어긋난 값, 같은 입력으로 확인한 수정 결과를 전달하세요. 이 기록이 있으면 프로젝트 팀은 검색과 화면 중 어디를 고칠지 합의할 수 있습니다.